<a href="https://colab.research.google.com/github/noor486/flyrank-ml-internship/blob/main/work/notebooks/w01_research_question.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/noor486/flyrank-ml-internship/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [7]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/noor486/flyrank-ml-internship"
REPO_DIR = "flyrank-ml-internship"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("../..")

print("Working dir:", os.getcwd())

# Generate outputs/model_results.json if it doesn't exist yet in this session
if not os.path.exists("outputs/model_results.json"):
    print("model_results.json not found — running the pipeline once to generate it...")
    subprocess.run([sys.executable, "scripts/run_all.py"], check=True)

assert os.path.exists("outputs/model_results.json"), "Pipeline ran but file still missing — check for errors above"
print("Ready.")

Working dir: /content/flyrank-ml-internship/flyrank-ml-internship
model_results.json not found — running the pipeline once to generate it...
Ready.


## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

## My Lane

**Lane 2: Refresh / Content Opportunity Scoring** — ranking pages by review priority
for refresh, expansion, protection, pruning, or monitoring.

Why: my Week 1–2 notebooks already built the exact core artifact this lane asks for —
a transparent baseline score vs. a learned model, evaluated with Precision@K. The gap
between them (shown below) is the exact question this lane wants investigated further,
not a coincidence I stumbled into.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import json

res = json.load(open("outputs/model_results.json"))
base = res["baseline"]["baseline_precision_at_50"]
rf = res["models"]["random_forest"]["precision_at_50"]

print(f"Baseline rule Precision@50: {base:.3f}  (~{round(base*50)} of top 50 correct)")
print(f"Random forest Precision@50: {rf:.3f}  (~{round(rf*50)} of top 50 correct)")
print(f"Gap: model beats baseline by {rf/base:.1f}x -- this is the opportunity this lane investigates")

Baseline rule Precision@50: 0.240  (~12 of top 50 correct)
Random forest Precision@50: 0.740  (~37 of top 50 correct)
Gap: model beats baseline by 3.1x -- this is the opportunity this lane investigates


## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

## The Question

**Research question:** Given a page's traffic, staleness, and ranking history, can we
rank pages by expected recovery value better than a simple "it's old and popular" rule?

**Unit of analysis:** a single page (`content_id`), scored at one point in time.

**Output:** a ranked queue of pages, highest expected-impact-if-refreshed first.

**The decision:** which pages a content team should refresh this sprint, given limited
editor capacity — they can't refresh everything at once.

**The action:** a content lead picks the top N pages off the ranked queue and assigns
them to writers for refresh, instead of picking pages by gut feel or "oldest first."

**Cost of a wrong call:**
- False positive (ranked high, refreshing doesn't help): wasted editor hours.
- False negative (a real decliner ranked low, ignored): continued traffic loss, caught late.

**Why data/ML helps:** staleness alone is a weak, single-variable proxy — my Week 1
discovery showed search volume barely predicts real traffic. A model combining several
signals has a real chance at ranking recovery value better than any one variable alone.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

n_rows = len(df)
n_unique_pages = df["content_id"].nunique()

print(f"Total rows: {n_rows}")
print(f"Unique content_id values: {n_unique_pages}")
print("Grain confirmed: one row per page" if n_rows == n_unique_pages else "WARNING: duplicates present, grain is not one row per page")

Total rows: 30000
Unique content_id values: 30000
Grain confirmed: one row per page


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

## Quick Look at the Data

Three real numbers backing this lane choice: the size of the opportunity, whether
staleness alone looks like a strong signal, and the verified baseline-vs-model gap.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd, numpy as np, json

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

# Number 1: how big is the actual opportunity?
declining_rate = df["is_declining_label"].mean()
print(f"Share of pages currently declining: {declining_rate:.1%}")

# Number 2: does staleness alone look like a strong signal on its own?
stale_corr = df["days_since_last_update"].corr(df["is_declining_label"])
print(f"Correlation between days_since_last_update and declining label: {stale_corr:.3f}")

# Number 3: verified reference numbers from outputs/model_results.json
res = json.load(open("outputs/model_results.json"))
base = res["baseline"]["baseline_precision_at_50"]
rf = res["models"]["random_forest"]["precision_at_50"]
print(f"Reference: hand-rule Precision@50 = {base:.3f}, model Precision@50 = {rf:.3f}")
print("(Client-holdout split -- whole clients excluded from training, not just rows)")

Share of pages currently declining: 54.2%
Correlation between days_since_last_update and declining label: 0.081
Reference: hand-rule Precision@50 = 0.240, model Precision@50 = 0.740
(Client-holdout split -- whole clients excluded from training, not just rows)


## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

## Careful Words

What I can claim: on this specific anonymized ~30k-row starter slice, using
client-holdout validation, a learned model ranks pages with far higher Precision@50
than a simple staleness rule. This is observed and directional, not universal.

What I can't claim:
- That refreshing a page *causes* recovery — that needs a real experiment, not this data.
- That this generalizes to the full ~79M-row warehouse without re-earning the result there.
- Anything about how Google's ranking algorithm works.

A known weakness I'm carrying forward: my current label
(`is_declining_label = trend_direction == "down"`) is a current-window proxy, not a
future outcome. A stronger version, for later weeks, would use prior-90-day features
to predict decline or recovery over the *next* 30 days instead.

No client names, domains, URLs, or raw queries appear anywhere in this notebook.

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
unsafe_keywords = ["client_name", "domain", "url", "raw_query", "title", "keyword_text"]
flagged = [c for c in df.columns if any(k in c.lower() for k in unsafe_keywords)]

print("Columns flagged for manual safety review:", flagged if flagged else "None found")
print("Columns actually used in this notebook:",
      ["content_id", "trend_direction", "days_since_last_update", "impressions_90d"])

Columns flagged for manual safety review: None found
Columns actually used in this notebook: ['content_id', 'trend_direction', 'days_since_last_update', 'impressions_90d']


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

## 5. Self-Check

- [x] Named a specific lane (Lane 2: Refresh / Content Opportunity Scoring)
- [x] Named the decision and the action
- [x] Named the cost of a wrong call in both directions
- [x] Backed the lane choice with real, verified numbers
- [x] Explained why this isn't just "train a model"
- [x] Named the current label's known weakness (current-window proxy)
- [x] Used observed/directional language throughout

In [12]:
import os

checks = {
    "starter data file exists": os.path.exists("data/raw/content_refresh_anonymized.csv"),
    "model results file exists": os.path.exists("outputs/model_results.json"),
}

for name, passed in checks.items():
    print(f"{'OK' if passed else 'MISSING'} -- {name}")

print("\nIf both show OK, this notebook can be rerun end-to-end from a clean clone.")

OK -- starter data file exists
OK -- model results file exists

If both show OK, this notebook can be rerun end-to-end from a clean clone.
